# Домашнее задание: валидация моделей генерации изображений

**Курс:** Валидация LLM  

## Цель задания

На лекции мы изучили, какими бывают автоматические метрики:
- референсные требуют ground truth изображений, с которыми сравниваются картинки, сгенерированные моделью;
- безреференсные не требуют реальных изображений, а оценивают соответствие сгенерированной картинки промпту.

В этом задании мы создадим небольшой пайплайн для оценки качества моделей T2I.

## Что мы сделаем

1. Сгенерируем изображения по сложным промптам
2. Используем FID, чтобы оценить качество или ... сгенерированных изображений
3. Используем CLIPScore и VQAScore для оценки соответствия изображений промптам

---

🔲 Помечены задания для выполнения

❓ Помечены вопросы для анализа

---

## Критерии оценки

Ваше решение будет оцениваться по следующим критериям:

### 1. Корректность кода (4 балла)
- Корректно рассчитаны статистики для FID (0.5 балла)
- Корректно написана формула для FID (0.5 балла)
- Корректно реализован аналогичный расчет FID (1 балл)  
- Корректно написана формула CLIPScore (0.5 балл)
- Корректно написана функция генерации вопросов для VQAScore (1.5 балла)

### 2. Анализ (6 баллов)
- Ответы на вопросы части 0 (0.5 балла)
- Ответы на вопросы части 1 (1.5 балла)
- Ответы на вопросы части 2.1 (2 балла)
- Ответы на вопросы части 2.2 (2 балла)

---

## Формат сдачи

Сохраните выполненный ноутбук (файл .ipynb) с названием `ДЗ_3_<Фамилия>_<ник в телеграмм>.ipynb` и пришлите в личные соощения телеграмм лектора Марии Лашиной (@m_rivaille).

**Успехов!**

In [ ]:
# Установка зависимостей
!pip install -q diffusers transformers torchmetrics datasets accelerate safetensors scipy pandas matplotlib

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import warnings
import random
import os
import gc

from diffusers import DiffusionPipeline

from scipy import linalg
from torchvision import models, transforms
from datasets import load_dataset

from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

from transformers import BlipProcessor, BlipForQuestionAnswering

warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
PyTorch: 2.8.0
GPU: CPU


In [ ]:
# Проверка памяти GPU
def check_gpu_memory():
    """Выводит информацию о памяти GPU"""
    if not torch.cuda.is_available():
        print("GPU не доступен")
        return

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    total = torch.cuda.get_device_properties(0).total_memory / 1024**2

    print(f"📊 GPU Memory Status:")
    print(f"   Total: {total:.0f} MB")
    print(f"   Allocated: {allocated:.1f} MB ({allocated/total*100:.1f}%)")
    print(f"   Reserved: {reserved:.1f} MB ({reserved/total*100:.1f}%)")
    print(f"   Free: {total - reserved:.1f} MB")

check_gpu_memory()

AttributeError: module 'torch.mps' has no attribute 'memory_allocated'

In [ ]:
# Очистка памяти перед загрузкой новой модели
def cleanup_memory():
    """Полная очистка памяти GPU"""
    # Удаляем все большие объекты
    for obj in gc.get_objects():
        try:
          if isinstance(obj, torch.Tensor):
              del obj
        except:
          pass

    # Очищаем кэш
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Сборщик мусора
    gc.collect()

    print("✅ Память очищена")
    check_gpu_memory()

cleanup_memory()

# Часть 0. Генерируем изображения

In [ ]:
# Сложные промпты
COMPLEX_PROMPTS = [
    "a red dragon breathing blue fire while flying over a crystal castle at midnight",
    "three astronauts playing chess with robots on Mars during a sandstorm",
    "a steampunk octopus wearing a Victorian hat and holding a golden compass",
    "a cyberpunk samurai meditating under a neon cherry blossom tree in rain",
    "a giant mushroom house with glowing windows surrounded by bioluminescent frogs"
]

In [ ]:
# Загрузка модели генерации
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to("cuda")

In [ ]:
# Генерация изображений
generated_images = []
used_prompts = []

print("Генерация изображений по сложным промптам...")
for i, prompt in enumerate(COMPLEX_PROMPTS):
    seed = 42 + i * 10
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe(prompt, num_inference_steps=30, generator=generator).images[0]
    generated_images.append(image)
    used_prompts.append(prompt)

print(f"✅ Сгенерировано {len(generated_images)} изображений.")

In [ ]:
# Визуализация
plt.figure(figsize=(15, 25))
for idx, img in enumerate(generated_images):
    plt.subplot(5, 1, idx+1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"{'-'*150}\nPrompt: {COMPLEX_PROMPTS[idx]}", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
output_dir = "/content/generated_images"
os.makedirs(output_dir, exist_ok=True)

# Сохраняем изображения с именами
saved_paths = []
print("Сохранение изображений...")
for idx, img in enumerate(generated_images):
    img_path = os.path.join(output_dir, f"image_{idx:03d}.png")
    img.save(img_path)
    saved_paths.append(img_path)

print(f"✅ Изображения сохранены в {output_dir}")

In [ ]:
cleanup_memory()

❓ Создайте Markdown ячейку ниже и ответьте на вопросы.

1. По вашему мнению, справилась ли модель с генерацией?

2. Какие изображения лично вам кажутся лучшими?

3. Ошиблась ли модель в каких-либо изображениях?

# Часть 1. Референсные метрики: FID

In [ ]:
# Загрузка Inception V3
inception_model = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT, transform_input=False)
inception_model.fc = torch.nn.Identity()
inception_model = inception_model.to("cuda")
inception_model.eval()

In [ ]:
# Трансформация для Inception
inception_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Функция получения активаций
def get_activations(images_list, model, batch_size=5):
    model.eval()
    activations = []
    with torch.no_grad():
        for i in range(0, len(images_list), batch_size):
            batch_imgs = images_list[i:i+batch_size]
            batch_tensor = torch.stack([inception_transform(img) for img in batch_imgs]).to("cuda")
            feat = model(batch_tensor)
            activations.append(feat.cpu().numpy())
    return np.vstack(activations)

In [ ]:
# Загрузка изображений для сравнения
dataset = load_dataset("cifar10", split="test")
real_images = [dataset[i]['img'] for i in range(len(generated_images))]

In [ ]:
# Расчет активаций
act_real = get_activations(real_images, inception_model)
act_fake = get_activations(generated_images, inception_model)

In [ ]:
# 🔲 Расчет статистик для FID
mu_real = np.mean(act_real, axis=0)
sigma_real = np.cov(act_real, rowvar=False)
mu_fake = np.mean(act_fake, axis=0)
sigma_fake = np.cov(act_fake, rowvar=False)

In [ ]:
# 🔲 Формула FID
# FID = ||μ_r - μ_g||² + Tr(Σ_r + Σ_g - 2(Σ_r Σ_g)^(1/2))
diff = mu_real - mu_fake
covmean, _ = linalg.sqrtm(sigma_real.dot(sigma_fake), disp=False)
if np.iscomplexobj(covmean):
    covmean = covmean.real
fid_score = np.sum(diff ** 2) + np.trace(sigma_real) + np.trace(sigma_fake) - 2 * np.trace(covmean)

print(f"\n=== FID Score ===")
print(f"FID: {fid_score:.2f}")
print("⚠️ Примечание: При N=5 это демо-значение, не статистически значимое")

In [ ]:
cleanup_memory()

❓ Создайте Markdown ячейку ниже и ответьте на вопросы.

Теория: FID сравнивает распределения признаков реальных и сгенерированных изображений.

1. Какой минимальный размер выборки рекомендуется для FID в научных статьях?

2. Почему сравнение изображений, сгенерированных Stable Diffusion, с изображениями из датасета CIFAR-10 некорректно? Укажите минимум 2 причины.

3. Что бы вы использовали в качестве альтернативного датасета для более адекватной оценки Stable Diffusion и почему?

Примеры популярных датасетов: COCO (2014), ImageNet Validation, LAION-Aesthetics, Flickr30k, Conceptual Captions, FFHQ, LSUN Bedrooms, LSUN Landscapes, WikiArt, Food-101, CheXpert, EuroSAT.

4. Какое еще свойство можно оценить для диффузионных моделей с помощью метрики FID, кроме качества изображений? Как это сделать?

🔲 Реализуйте пункт 3 или 4 и проанализируйте результаты.

In [ ]:
# 🔲 Аналогичный расчёт FID (пункт 3: альтернативный датасет)
# Используем CIFAR-100 вместо CIFAR-10 как альтернативный референсный датасет
# (для продакшена лучше COCO, ImageNet или доменно-специфичный датасет)

dataset_alt = load_dataset("cifar100", split="test")
real_images_alt = [dataset_alt[i]['img'] for i in range(len(generated_images))]

act_real_alt = get_activations(real_images_alt, inception_model)
act_fake_alt = get_activations(generated_images, inception_model)

mu_real_alt = np.mean(act_real_alt, axis=0)
sigma_real_alt = np.cov(act_real_alt, rowvar=False)
mu_fake_alt = np.mean(act_fake_alt, axis=0)
sigma_fake_alt = np.cov(act_fake_alt, rowvar=False)

diff_alt = mu_real_alt - mu_fake_alt
covmean_alt, _ = linalg.sqrtm(sigma_real_alt.dot(sigma_fake_alt), disp=False)
if np.iscomplexobj(covmean_alt):
    covmean_alt = covmean_alt.real
fid_score_alt = np.sum(diff_alt ** 2) + np.trace(sigma_real_alt) + np.trace(sigma_fake_alt) - 2 * np.trace(covmean_alt)

print("=== FID с альтернативным датасетом (CIFAR-100) ===")
print(f"FID (CIFAR-10):  {fid_score:.2f}")
print(f"FID (CIFAR-100): {fid_score_alt:.2f}")
print("При N=5 оба значения демонстрационные; для научной оценки нужна выборка 10k+.")

# Часть 2. Референсные метрики: CLIPScore, VQAScore

## 2.1. CLIP

In [ ]:
# Загрузка CLIP модели
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).to("cuda")
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

In [ ]:
# Подготовка данных
inputs = clip_processor(text=used_prompts, images=generated_images, return_tensors="pt", padding=True).to("cuda")

In [ ]:
# Получение эмбеддингов
with torch.no_grad():
    outputs = clip_model(**inputs)
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds

In [ ]:
# 🔲 Рассчет CLIP Score
# CLIPScore = косинусное сходство между нормализованными эмбеддингами изображения и текста
clip_scores = F.cosine_similarity(image_embeds, text_embeds, dim=1)

print(f"\n=== CLIP Score Results ===")
print(f"Средний CLIP Score: {clip_scores.mean().item():.2f}")
for i, score in enumerate(clip_scores):
    print(f"Image {i}: {score.item():.2f} | Prompt: {used_prompts[i]}")

In [ ]:
cleanup_memory()

❓ Создайте Markdown ячейку ниже и ответьте на вопросы.

1. Какой средний CLIP Score вы получили? Что это говорит о качестве генерации?

2. Какой промпт получил наивысший балл? Какой промпт получил низший балл? Соответствует ли это вашему визуальному восприятию?

3. Есть ли примеры, где балл CLIP Score не совпал с ожиданиями? Почему это произошло?

4. Учитывая ответ на предыдущий вопрос, как бы вы дополнили оценку качества модели? Подробно опишите минимум один подход, кроме приведенных в этом задании.

## 2.2. VQAScore

In [ ]:
# Загрузка VQA модели
blip_vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
blip_vqa_model = BlipForQuestionAnswering.from_pretrained(
    "Salesforce/blip-vqa-base"
).to("cuda")
blip_vqa_model.eval()

In [ ]:
# Функция для создания вопросов для VQA
def create_vqa_questions(prompt):
    """
    Создает VQA вопросы на основе оригинального промпта
    """
    questions = []
    answers = []

    # Простой вопрос для начала
    questions.append(f"Does this picture show '{prompt}'?")
    answers.append('yes')

    # Ключевые слова
    keywords = ['dragon', 'fire', 'dog', 'castle', 'astronaut', 'cat', 'octopus', 'teaspoon', 'car', 'spaceship', 'unicorn',
                'samurai', 'wizard', 'frog', 'castle', 'car', 'tree', 'house', 'robot']

    prompt_lower = prompt.lower()

    # Добавляем в вопросы все слова, которые есть в промпте, и 3 вопроса про слова, которых нет в промпте
    keywords_not_in_prompt = []
    for keyword in keywords:
        if keyword in prompt_lower:
            questions.append(f"Is there a {keyword} in the image?")
            answers.append('yes')
        else:
            keywords_not_in_prompt.append(keyword)

    keywords_not_in_prompt_to_add = random.sample(keywords_not_in_prompt, 3)
    for keyword in keywords_not_in_prompt_to_add:
        questions.append(f"Is there a {keyword} in the image?")
        answers.append('no')

    # 🔲 Добавьте еще вопросы-ответы минимум двух типов

    # Тип 1: вопрос о количестве (counting)
    questions.append("How many main subjects are in the image?")
    answers.append("one")

    # Тип 2: вопрос о цвете/атрибуте (извлекаем из промпта)
    color_words = ['red', 'blue', 'green', 'golden', 'neon', 'glowing', 'crystal', 'gold', 'white', 'black']
    found_color = next((c for c in color_words if c in prompt_lower), None)
    if found_color:
        questions.append("What color is dominant in the image?")
        answers.append(found_color)
    else:
        questions.append("Is the image in color?")
        answers.append("yes")

    # Тип 3: вопрос о сцене (обстановка/время)
    if 'night' in prompt_lower or 'midnight' in prompt_lower:
        questions.append("Is it night in the image?")
        answers.append("yes")
    else:
        questions.append("Is it daytime in the image?")
        answers.append("yes")

    return questions, answers

In [ ]:
# Оценка VQA по изображениям
vqa_results = []

print("Выполнение VQA оценки...")
for idx, img_path in enumerate(saved_paths):
    img = Image.open(img_path).convert("RGB")
    prompt = used_prompts[idx]

    # Создание вопросов для этого изображения
    questions, answers = create_vqa_questions(prompt)

    image_scores = []

    for question, answer in zip(questions, answers):
        # Подготовка входа для VQA
        inputs = blip_vqa_processor(
            img,
            question,
            return_tensors="pt"
        ).to("cuda")

        # Генерация ответа
        with torch.no_grad():
            outputs = blip_vqa_model.generate(**inputs)

        # Декодирование ответа
        model_answer = blip_vqa_processor.decode(outputs[0], skip_special_tokens=True)

        # Оценка ответа
        if answer in model_answer.lower():
            score = 1.0
        else:
            score = 0.0

        image_scores.append({
            'question': question,
            'answer': answer,
            'model_answer': model_answer,
            'score': score
        })

        print(f"  Q: {question}")
        print(f"  A: {answer}")
        print(f"  MA: {model_answer} (score: {score})")

    # Средний score для этого изображения
    avg_image_score = sum(item['score'] for item in image_scores) / len(image_scores)

    vqa_results.append({
        'image_idx': idx,
        'prompt': prompt[:50],
        'questions': image_scores,
        'avg_score': avg_image_score
    })

    print(f"Image {idx} VQA Score: {avg_image_score:.2f}\n")

In [ ]:
overall_vqa_score = sum(r['avg_score'] for r in vqa_results) / len(vqa_results)

print(f"{'='*60}")
print(f"=== BLIP VQA Results ===")
print(f"{'='*60}")
print(f"Средний VQA Score: {overall_vqa_score:.4f}")
print(f"{'='*60}")

In [ ]:
cleanup_memory()

❓ Создайте Markdown ячейку ниже и ответьте на вопросы.

1. Какой средний VQA Score вы получили? Что это говорит о качестве генерации?

2. Какой промпт получил наивысший балл? Какой промпт получил низший балл? Соответствует ли это вашему визуальному восприятию?

3. Есть ли примеры, где балл VQA Score не совпал с ожиданиями? Почему это произошло?

4. Какие ограничения есть у VQA-based оценки генеративных моделей?

In [ ]:
# @title
